# **Built-in LSTM**

In [5]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Concatenate, Flatten, Dropout

# -------------------------------
# 1. Load and preprocess data
# -------------------------------
panel_path = "/content/panel_for_model.csv"  # Update path
panel = pd.read_csv(panel_path, parse_dates=['Date'])
features = ['active','new_confirmed','daily_tests','daily_vax','vax_rate']
seq_len = 14

X_list, region_list, y_list = [], [], []
states = panel['state'].unique()

for st in states:
    sub = panel[panel['state']==st].sort_values('Date').reset_index(drop=True)
    if len(sub) <= seq_len:
        continue
    arr = sub[features].values
    for i in range(len(sub)-seq_len):
        X_list.append(arr[i:i+seq_len])
        region_list.append(int(sub.loc[i+seq_len, 'state_id']))
        y_list.append(sub.loc[i+seq_len, 'active'])

X = np.stack(X_list)
y = np.array(y_list)
region_idx = np.array(region_list)

# Take 10% of data for demo
n_samples = int(0.1 * len(X))
X_sample = X[:n_samples]
region_sample = region_idx[:n_samples]
y_sample = y[:n_samples]

# -------------------------------
# 2. Build region-aware LSTM model
# -------------------------------
seq_len = X.shape[1]
n_features = X.shape[2]
n_regions = int(region_idx.max()) + 1
embed_dim = 8

# Inputs
seq_in = Input(shape=(seq_len, n_features), name='seq_in')
region_in = Input(shape=(1,), dtype='int32', name='region_in')

# LSTM encoder
x = LSTM(64, activation='tanh', return_sequences=True, name='lstm_enc_seq')(seq_in)
x_last = LSTM(64, activation='tanh', name='lstm_enc')(x)
x_drop = Dropout(0.2)(x_last)

# Region embedding
emb = Embedding(input_dim=n_regions, output_dim=embed_dim, name='region_emb')(region_in)
embf = Flatten()(emb)

# Concatenate features
concat = Concatenate()([x_drop, embf])
dense1 = Dense(64, activation='relu', name='dense1')(concat)
dense_drop = Dropout(0.2)(dense1)
out = Dense(1, activation='sigmoid', name='output')(dense_drop)

# Model compilation
model = Model(inputs=[seq_in, region_in], outputs=out)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# -------------------------------
# 3. Intermediate models to inspect outputs
# -------------------------------
# LSTM outputs for all timesteps
model_lstm_seq = Model(inputs=seq_in, outputs=x)
# Final LSTM hidden state
model_lstm_last = Model(inputs=seq_in, outputs=x_last)
# Dense layer output before final prediction
model_dense = Model(inputs=[seq_in, region_in], outputs=dense1)

# -------------------------------
# 4. Get outputs for sample
# -------------------------------
lstm_seq_out = model_lstm_seq.predict(X_sample)
lstm_last_out = model_lstm_last.predict(X_sample)
dense_out = model_dense.predict([X_sample, region_sample])
full_out = model.predict([X_sample, region_sample])

# Display shapes
print("LSTM sequence output shape:", lstm_seq_out.shape)
print("LSTM last hidden state shape:", lstm_last_out.shape)
print("Dense layer output shape:", dense_out.shape)
print("Full model output shape:", full_out.shape)

# -------------------------------
# 5. Optional: inspect first sample
# -------------------------------
print("\nExample outputs for first sample:")
print("LSTM sequence output:\n", lstm_seq_out[0])
print("LSTM last hidden state:\n", lstm_last_out[0])
print("Dense layer output:\n", dense_out[0])
print("Final prediction:\n", full_out[0])

# -------------------------------
# 6. Custom LSTM with gate outputs (for demonstration)
# -------------------------------
class LSTMWithGates(Layer):
    def __init__(self, units, **kwargs):
        super(LSTMWithGates, self).__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        input_dim = input_shape[-1]
        self.W = self.add_weight(shape=(input_dim, 4*self.units),
                                 initializer='glorot_uniform', name='W')
        self.U = self.add_weight(shape=(self.units, 4*self.units),
                                 initializer='orthogonal', name='U')
        self.b = self.add_weight(shape=(4*self.units,), initializer='zeros', name='b')
        super(LSTMWithGates, self).build(input_shape)

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        h_t = tf.zeros((batch_size, self.units))
        c_t = tf.zeros((batch_size, self.units))
        h_seq, f_seq, i_seq, o_seq = [], [], [], []
        inputs_unstacked = tf.unstack(inputs, axis=1)

        for x_t in inputs_unstacked:
            z = tf.matmul(x_t, self.W) + tf.matmul(h_t, self.U) + self.b
            i, f, c_bar, o = tf.split(z, num_or_size_splits=4, axis=1)
            i_t = tf.sigmoid(i)
            f_t = tf.sigmoid(f)
            o_t = tf.sigmoid(o)
            c_bar_t = tf.tanh(c_bar)
            c_t = f_t * c_t + i_t * c_bar_t
            h_t = o_t * tf.tanh(c_t)
            h_seq.append(h_t)
            f_seq.append(f_t)
            i_seq.append(i_t)
            o_seq.append(o_t)

        h_seq = tf.stack(h_seq, axis=1)
        f_seq = tf.stack(f_seq, axis=1)
        i_seq = tf.stack(i_seq, axis=1)
        o_seq = tf.stack(o_seq, axis=1)
        return h_seq, c_t, i_seq, f_seq, o_seq

# Sample input for gate demo
n_samples_demo = 2
seq_len_demo = 5
n_features_demo = 3
X_demo = np.random.rand(n_samples_demo, seq_len_demo, n_features_demo).astype(np.float32)

inputs_demo = Input(shape=(seq_len_demo, n_features_demo))
h_seq, c_last, i_seq, f_seq, o_seq = LSTMWithGates(4)(inputs_demo)
out_demo = Dense(1, activation='sigmoid')(h_seq[:, -1, :])
model_demo = Model(inputs=inputs_demo, outputs=[h_seq, c_last, i_seq, f_seq, o_seq, out_demo])
model_demo.compile(optimizer='adam', loss='mse')

# Forward pass
h_seq_out, c_last_out, i_seq_out, f_seq_out, o_seq_out, final_out = model_demo.predict(X_demo)
print("\nCustom LSTM demo outputs:")
print("Hidden states h_t shape:", h_seq_out.shape)
print("Last cell state C_t shape:", c_last_out.shape)
print("Input gate i_t shape:", i_seq_out.shape)
print("Forget gate f_t shape:", f_seq_out.shape)
print("Output gate o_t shape:", o_seq_out.shape)
print("Final prediction:", final_out)

print("\nExample gates for first sample:")
for t in range(seq_len_demo):
    print(f"Time {t}: i_t={i_seq_out[0,t]}, f_t={f_seq_out[0,t]}, o_t={o_seq_out[0,t]}")


79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
LSTM sequence output shape: (2511, 14, 64)
LSTM last hidden state shape: (2511, 64)
Dense layer output shape: (2511, 64)
Full model output shape: (2511, 1)

Example outputs for first sample:
LSTM sequence output:
 [[-6.37150854e-02 -2.96820551e-02 -3.72137204e-02  1.84938610e-02
   4.98784557e-02 -1.62712280e-02 -2.20968369e-02 -1.57374740e-02
   8.37880280e-03  2.37156097e-02  5.58300316e-02  1.47002665e-02
   2.87642870e-02 -1.17535125e-02  1.86051168e-02  1.70516763e-02
   8.23094323e-03 -2.14761030e-02  2.60138288e-02  2.13211086e-02
   1.31027140e-02  2.58139372e-02  4.59437445e-02 -3.07908840e-03
   1.02427620e-02 -1.33576030e-02 -7.66137196e-03  7.97677785e-03
  -3.42006460e-02 -2.70540696e-02  4.76565287e-02 -2.57296748e-02
  -2.72977930e-02 -1.11455247e-02  2.08933670e-02  3.79118174e-02
  -3.25877070e-02  4.50685993e-02 

# **Layers manually using formulas**

In [6]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Flatten
from tensorflow.keras.initializers import GlorotUniform, Orthogonal, Zeros

# -------------------------------
# 1. Load and preprocess data
# -------------------------------
panel_path = "/content/panel_for_model.csv"  # update path
panel = pd.read_csv(panel_path, parse_dates=['Date'])
features = ['active','new_confirmed','daily_tests','daily_vax','vax_rate']
seq_len = 14

X_list, region_list, y_list = [], [], []
states = panel['state'].unique()

for st in states:
    sub = panel[panel['state']==st].sort_values('Date').reset_index(drop=True)
    if len(sub) <= seq_len:
        continue
    arr = sub[features].values
    for i in range(len(sub)-seq_len):
        X_list.append(arr[i:i+seq_len])
        region_list.append(int(sub.loc[i+seq_len, 'state_id']))
        y_list.append(sub.loc[i+seq_len, 'active'])

X = np.stack(X_list).astype(np.float32)
y = np.array(y_list, dtype=np.float32)
region_idx = np.array(region_list, dtype=np.int32)

# Take small subset for demo
n_samples = int(0.1 * len(X))
X_sample = X[:n_samples]
region_sample = region_idx[:n_samples]
y_sample = y[:n_samples]

# -------------------------------
# 2. Hyperparameters
# -------------------------------
n_features = X.shape[2]
seq_len = X.shape[1]
n_regions = int(region_idx.max()) + 1
embed_dim = 8
lstm_units = 16
dense_units = 32

# -------------------------------
# 3. Define weights manually
# -------------------------------
initializer_W = GlorotUniform()
initializer_U = Orthogonal()
initializer_b = Zeros()

# LSTM weights
W = tf.Variable(initializer_W(shape=(n_features, 4*lstm_units)))
U = tf.Variable(initializer_U(shape=(lstm_units, 4*lstm_units)))
b = tf.Variable(initializer_b(shape=(4*lstm_units,)))

# Region embedding weights
region_emb = tf.Variable(initializer_W(shape=(n_regions, embed_dim)))

# Dense layer weights
W_dense1 = tf.Variable(initializer_W(shape=(lstm_units + embed_dim, dense_units)))
b_dense1 = tf.Variable(initializer_b(shape=(dense_units,)))

W_out = tf.Variable(initializer_W(shape=(dense_units, 1)))
b_out = tf.Variable(initializer_b(shape=(1,)))

# -------------------------------
# 4. Forward pass using formulas
# -------------------------------
def manual_lstm_region_forward(X_seq, region_idx):
    batch_size = X_seq.shape[0]

    # Initialize LSTM states
    h_t = tf.zeros((batch_size, lstm_units))
    c_t = tf.zeros((batch_size, lstm_units))
    h_seq = []

    # Iterate over time steps
    for t in range(seq_len):
        x_t = X_seq[:, t, :]
        z = tf.matmul(x_t, W) + tf.matmul(h_t, U) + b

        # Split gates
        i, f, c_bar, o = tf.split(z, num_or_size_splits=4, axis=1)

        # Activations
        i_t = tf.sigmoid(i)
        f_t = tf.sigmoid(f)
        o_t = tf.sigmoid(o)
        c_bar_t = tf.tanh(c_bar)

        # Cell and hidden state updates
        c_t = f_t * c_t + i_t * c_bar_t
        h_t = o_t * tf.tanh(c_t)

        h_seq.append(h_t)

    h_seq = tf.stack(h_seq, axis=1)  # [batch, seq_len, lstm_units]
    h_last = h_seq[:, -1, :]         # final hidden state

    # Region embedding
    region_vec = tf.nn.embedding_lookup(region_emb, region_idx)  # [batch, embed_dim]

    # Concatenate LSTM hidden state and region embedding
    concat = tf.concat([h_last, region_vec], axis=1)  # [batch, lstm_units + embed_dim]

    # Dense layer
    dense1 = tf.nn.relu(tf.matmul(concat, W_dense1) + b_dense1)

    # Output layer
    out = tf.sigmoid(tf.matmul(dense1, W_out) + b_out)

    return h_seq, h_last, dense1, out

# -------------------------------
# 5. Forward pass on sample
# -------------------------------
h_seq_out, h_last_out, dense_out, final_out = manual_lstm_region_forward(X_sample, region_sample)

print("LSTM sequence output shape:", h_seq_out.shape)
print("LSTM last hidden state shape:", h_last_out.shape)
print("Dense layer output shape:", dense_out.shape)
print("Final prediction shape:", final_out.shape)

# Inspect first sample
print("\nFirst sample outputs:")
print("LSTM last hidden state:\n", h_last_out[0])
print("Dense layer output:\n", dense_out[0])
print("Final prediction:\n", final_out[0])


LSTM sequence output shape: (2511, 14, 16)
LSTM last hidden state shape: (2511, 16)
Dense layer output shape: (2511, 32)
Final prediction shape: (2511, 1)

First sample outputs:
LSTM last hidden state:
 tf.Tensor(
[ 0.0322938   0.123616    0.13984714 -0.12152223  0.17277099 -0.01142242
  0.12084977  0.00735518  0.11006281  0.03340016 -0.15367256 -0.02469165
  0.12501404 -0.09152513 -0.0742675   0.05289108], shape=(16,), dtype=float32)
Dense layer output:
 tf.Tensor(
[0.         0.07449602 0.00316852 0.04005207 0.12499226 0.11846564
 0.         0.         0.         0.09705772 0.10170769 0.25566223
 0.         0.         0.         0.         0.13301438 0.1101073
 0.05602053 0.14661473 0.22064836 0.         0.08095217 0.00537006
 0.21278936 0.01918391 0.19462217 0.         0.05482624 0.
 0.23490116 0.03453381], shape=(32,), dtype=float32)
Final prediction:
 tf.Tensor([0.5148014], shape=(1,), dtype=float32)
